In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, col, current_timestamp, expr, struct, lit
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.appName("DataGen").getOrCreate()

# Clickstream (10M events, nested JSON)
spark.range(1, 10000000) \
    .withColumn("event_id", col("id")) \
    .withColumn("user_id", (rand() * 10000).cast(IntegerType())) \
    .withColumn("event_ts", current_timestamp() - expr(f"INTERVAL {7} DAY") * rand() * 7) \
    .withColumn("page_data", struct(lit("home").alias("page"), (rand() * 100).alias("time_spent"))) \
    .select("event_id", "user_id", "event_ts", "page_data") \
    .write.mode("overwrite").json("data/clickstream_raw/")

# IoT data (500K records, CSV)
spark.range(1, 500000) \
    .selectExpr(
        "id as sensor_id",
        "cast(rand() * 10000 as int) as warehouse_id",
        "rand() * 50 as temp_celsius",
        "rand() * 80 as humidity_pct"
    ) \
    .coalesce(1).write.mode("overwrite").csv("data/iot_raw/", header=True)